# 6 · Execution and impact

What happens between the decision and the fill.

A historical print tells you the price, never what your order would have
done to it. Here the same seed runs twice — once with your orders and once
without — so every fill is priced against the market where you never traded.

That counterfactual is what arrival price, VWAP and fitted impact models
approximate. Here it is measured directly.

In [1]:
import pretium as pt

universe = pt.Universe.random(20, seed=111)

execution = pt.tca.analyse(pt.baselines.Momentum(),
                           seed=7, universe=universe, days=3)

print("fills           :", len(execution.fills))
print("partial fills   :", len(execution.partial_fills()))
print("shortfall (bps) :", round(execution.shortfall_bps(), 3))

fills           : 149
partial fills   : 0
shortfall (bps) : 3.706


## The counterfactual run

`actual_final` is the closing price of each instrument. `baseline_final` is
the same market with the agent's orders removed: same seed, same draws,
everything else identical. Where they differ, that difference is the
agent's footprint.

In [2]:
tickers = [i.ticker for i in universe]
actual, baseline = execution.actual_final, execution.baseline_final

print(f"{'ticker':8s} {'with orders':>13s} {'without':>13s} {'difference':>12s}")
for t, a, b in zip(tickers, actual, baseline):
    if a != b:
        print(f"{t:8s} {a:13.4f} {b:13.4f} {a - b:12.4f}")

untouched = sum(1 for a, b in zip(actual, baseline) if a == b)
print(f"\n{untouched} of {len(tickers)} names were not moved at all.")

ticker     with orders       without   difference
AAA           193.5400      193.1700       0.3700
AAB            11.8600       11.8800      -0.0200
AAC            23.6900       23.6500       0.0400
AAG           276.9900      276.3300       0.6600
AAJ           128.4200      128.4100       0.0100
AAL            53.9200       53.9100       0.0100
AAM           101.0500      101.0900      -0.0400
AAO            10.5100       10.5600      -0.0500
AAP           137.8800      144.1200      -6.2400
AAR             4.9400        4.8400       0.1000
AAT           493.7800      493.8600      -0.0800

9 of 20 names were not moved at all.


## Which names moved

`moved()` gives the price displacement the agent caused, per instrument,
measured as the difference between the two runs.

In [3]:
moved = execution.moved()
worst = sorted(moved.items(), key=lambda kv: -abs(kv[1]))[:6]

print(f"{'ticker':8s} {'price moved':>13s} {'impact (bps)':>14s}")
for ticker, delta in worst:
    print(f"{ticker:8s} {delta:13.4f} {execution.impact_bps(ticker):14.3f}")

ticker     price moved   impact (bps)
AAP          -432.9725       -432.973
AAR           206.6116        206.612
AAO           -47.3485        -47.348
AAG            23.8845         23.884
AAA            19.1541         19.154
AAC            16.9133         16.913


## Where the cost fell

`by_step` attributes P&L across decision points and `by_ticker` across
names. Cost concentrated in a few steps says something about the
schedule.

In [4]:
steps = execution.by_step()
print("by step:")
for step, value in steps:
    bar = "#" * int(min(40, abs(value) / max(1, max(abs(v) for _, v in steps)) * 40))
    print(f"  step {step:3d} {value:12,.0f}  {bar}")

by step:
  step   6          948  ###############################
  step   7          133  ####
  step   8          205  ######
  step   9          474  ###############
  step  10         -235  #######
  step  11          979  ################################
  step  12          -54  #
  step  13          437  ##############
  step  14         -454  ###############
  step  15           72  ##
  step  16        1,206  ########################################
  step  17         -921  ##############################


In [5]:
by_ticker = execution.by_ticker()
top = sorted(by_ticker.items(), key=lambda kv: -abs(kv[1]))[:6]
print("largest contributions by name:")
for ticker, value in top:
    print(f"  {ticker:8s} {value:12,.0f}")

largest contributions by name:
  AAA            -1,536
  AAR             1,312
  AAG            -1,221
  AAD               617
  AAS               585
  AAP               527


## Partial fills

An order asking for more than the book holds at a price does not silently
receive it. Queue position and depth decide what you actually get.

In [6]:
if execution.partial_fills():
    print(f"{len(execution.partial_fills())} partial fills")
    for f in execution.partial_fills()[:5]:
        print("  ", f)
else:
    print("No partial fills at this size — the book absorbed every order.")
    print("Raising participation or size is how you find the edge of that;")
    print("the whole point is that the edge exists and is measurable.")

No partial fills at this size — the book absorbed every order.
Raising participation or size is how you find the edge of that;
the whole point is that the edge exists and is measurable.


## Comparing two algorithms

Same market, same seed, with the counterfactual computed for each.

In [7]:
candidates = {
    "momentum":       pt.baselines.Momentum(),
    "mean reversion": pt.baselines.MeanReversion(),
    "buy and hold":   pt.baselines.BuyAndHold(),
}

print(f"{'algorithm':16s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for name, agent in candidates.items():
    ex = pt.tca.analyse(agent, seed=7, universe=universe, days=3)
    print(f"{name:16s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")

algorithm         shortfall bps   fills  partials
momentum                  3.706     149         0
mean reversion           -0.586     151         0
buy and hold             13.191      20         0


## Provenance

An execution result carries the seed and model fingerprint, so a TCA number
can be cited.

In [8]:
print("seed             :", execution.seed)
print("model fingerprint:", execution.model_fingerprint)

seed             : 7
model fingerprint: pt-v3


## Caveats

**Volume changes are the model's structural failure.** Their
autocorrelation misses its real-market band by about 13.7
seed-standard-deviations, so a volume forecast here is never wrong twice
running. An execution algorithm tested against this market faces an easier
scheduling problem than the one it was written for. Treat schedule-shape
conclusions with suspicion; depth, queue and impact conclusions are sound.

**Single venue, no latency.** One book per name, orders arrive instantly,
and no strategic counterparties adapt to you.

Full documentation: <https://simoncoombes.github.io/pretium/>